In [10]:
#dashboard dataset
#all featres included
import pandas as pd
dashboard_df = pd.read_csv("Cleaned_Superstore.csv", encoding="latin1")

dashboard_df.to_csv("Dashboard_Data.csv", index=False)

In [12]:
#meachine learning dataset
#removing unnecesssary features
# Drop unnecessary columns
df=pd.read_csv("Cleaned_Superstore.csv", encoding="latin1")
df = df.drop(columns=[
    "Row ID",
    "Order ID",
    "Order Date",
    "Ship Date",
    "Customer ID",
    "Customer Name",
    "Country",
    "Postal Code",
    "Product ID",
    "Product Name",
    "Profit",
    "Day",
    "Day Name",
    "Quarter",
    "Month Name"
])

df.head()

,Ship Mode,Segment,City,State,Region,Category,Sub-Category,Sales,Quantity,Discount,Year,Month,Shipping Days
0,Second Class,Consumer,Henderson,Kentucky,South,Furniture,Bookcases,261.9600,2,0.00,2016,11,3
1,Second Class,Consumer,Henderson,Kentucky,South,Furniture,Chairs,731.9400,3,0.00,2016,11,3
2,Second Class,Corporate,Los Angeles,California,West,Office Supplies,Labels,14.6200,2,0.00,2016,6,4
3,Standard Class,Consumer,Fort Lauderdale,Florida,South,Furniture,Tables,957.5775,5,0.45,2015,10,7
4,Standard Class,Consumer,Fort Lauderdale,Florida,South,Office Supplies,Storage,22.3680,2,0.20,2015,10,7


In [14]:
df.to_csv("ML_Data.csv", index=False)

In [15]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import RandomForestRegressor

from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [17]:
#load the dataset
df = pd.read_csv("ML_Data.csv")

df.head()

,Ship Mode,Segment,City,State,Region,Category,Sub-Category,Sales,Quantity,Discount,Year,Month,Shipping Days
0,Second Class,Consumer,Henderson,Kentucky,South,Furniture,Bookcases,261.9600,2,0.00,2016,11,3
1,Second Class,Consumer,Henderson,Kentucky,South,Furniture,Chairs,731.9400,3,0.00,2016,11,3
2,Second Class,Corporate,Los Angeles,California,West,Office Supplies,Labels,14.6200,2,0.00,2016,6,4
3,Standard Class,Consumer,Fort Lauderdale,Florida,South,Furniture,Tables,957.5775,5,0.45,2015,10,7
4,Standard Class,Consumer,Fort Lauderdale,Florida,South,Office Supplies,Storage,22.3680,2,0.20,2015,10,7


In [18]:
#checking the dataset
print(df.shape)

df.info()

(9994, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Ship Mode      9994 non-null   object 
 1   Segment        9994 non-null   object 
 2   City           9994 non-null   object 
 3   State          9994 non-null   object 
 4   Region         9994 non-null   object 
 5   Category       9994 non-null   object 
 6   Sub-Category   9994 non-null   object 
 7   Sales          9994 non-null   float64
 8   Quantity       9994 non-null   int64  
 9   Discount       9994 non-null   float64
 10  Year           9994 non-null   int64  
 11  Month          9994 non-null   int64  
 12  Shipping Days  9994 non-null   int64  
dtypes: float64(2), int64(4), object(7)
memory usage: 1015.1+ KB


In [19]:
#defining input and target features
X = df.drop("Sales", axis=1)

y = df["Sales"]

In [20]:
#identify column types
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numerical_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print(categorical_features)

print(numerical_features)

['Ship Mode', 'Segment', 'City', 'State', 'Region', 'Category', 'Sub-Category']
['Quantity', 'Discount', 'Year', 'Month', 'Shipping Days']


In [21]:
#building preprocessing pipeline
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [22]:
#splitting dataset into train and test data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [23]:
#building the models
models = {

    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(random_state=42),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )

}

In [24]:
#training and evaluating the models
results = []

trained_models = {}

for name, model in models.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    predictions = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))

    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": name,
        "MAE": round(mae,2),
        "RMSE": round(rmse,2),
        "R2 Score": round(r2,4)
    })

    trained_models[name] = pipe

In [25]:
#comparing the models
results_df = pd.DataFrame(results)

results_df.sort_values(
    by="R2 Score",
    ascending=False
)

,Model,MAE,RMSE,R2 Score
3,Gradient Boosting,200.68,686.08,0.2031
0,Linear Regression,242.70,697.37,0.1767
2,Random Forest,200.77,709.93,0.1468
1,Decision Tree,235.65,812.00,-0.1162


In [26]:
#selecting the best model
best_model_name = results_df.sort_values(
    by="R2 Score",
    ascending=False
).iloc[0]["Model"]

best_model = trained_models[best_model_name]

print("Best Model:", best_model_name)

Best Model: Gradient Boosting


In [27]:
#save the model
joblib.dump(
    best_model,
    "sales_prediction_model.pkl"
)

print("Model Saved Successfully!")

Model Saved Successfully!


In [28]:
#test prediction
sample = X_test.iloc[[0]]

prediction = best_model.predict(sample)

print("Predicted Sales :", prediction[0])

print("Actual Sales :", y_test.iloc[0])

Predicted Sales : 224.2808088225423
Actual Sales : 563.808


In [29]:
#save models comparision
results_df.to_csv(
    "model_comparison.csv",
    index=False
)